<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/07_Ethique_IA_Medicale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. Éthique et Responsabilité de l'IA en Médecine

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

Ce cours vous permettra de:
- Comprendre les enjeux éthiques de l'IA en médecine
- Identifier les biais et risques des systèmes d'IA médicale
- Analyser des cas pratiques de déploiement d'IA
- Développer une réflexion critique sur l'utilisation responsable de l'IA

**Format:** Ce cours est principalement basé sur la discussion et la réflexion, avec quelques démonstrations de code pour illustrer les concepts.

## 1. Les Quatre Principes de l'Éthique Médicale appliqués à l'IA

### 1.1 Bienfaisance (Faire le bien)

**Question:** L'IA améliore-t-elle vraiment les soins?

**Bénéfices potentiels:**
- Détection précoce de maladies
- Réduction des erreurs diagnostiques
- Accès aux soins dans les zones sous-dotées
- Personnalisation des traitements

**Mais attention:**
- L'IA doit prouver son bénéfice clinique (pas juste technique)
- Les études doivent être rigoureuses et indépendantes
- L'impact sur le workflow doit être positif

### 1.2 Non-malfaisance (Ne pas nuire)

**Risques de l'IA médicale:**
- Faux négatifs (pathologies manquées)
- Faux positifs (examens/traitements inutiles)
- Sur-diagnostic et sur-traitement
- Déshumanisation de la relation médecin-patient

### 1.3 Autonomie (Respect du patient)

**Questions importantes:**
- Le patient sait-il qu'une IA est utilisée?
- Peut-il refuser l'analyse IA?
- Comprend-il les limites de l'IA?
- Ses données sont-elles protégées?

### 1.4 Justice (Équité)

**Enjeux d'équité:**
- Accès égal à l'IA (riches vs pauvres, urbain vs rural)
- Biais algorithmiques (performance inégale selon les groupes)
- Coût des technologies IA
- Formation des professionnels de santé

## 2. Les Biais en IA Médicale

### 2.1 Qu'est-ce qu'un biais?

Un **biais** est une erreur systématique qui fait que le modèle d'IA performe différemment selon les groupes de patients.

### 2.2 Types de biais

**Biais de représentation:**
- Dataset majoritairement caucasien → moins performant sur autres ethnicités
- Données principalement d'hôpitaux universitaires → moins performant en milieu rural
- Peu de cas pédiatriques → mauvaise performance chez l'enfant

**Biais de mesure:**
- Labels créés par des radiologues fatigués
- Diagnostics plus fréquents dans certains hôpitaux
- Qualité d'image variable selon l'équipement

**Biais de confirmation:**
- L'IA renforce les préjugés existants
- Cercle vicieux: IA biaisée → décisions biaisées → nouvelles données biaisées

### 2.3 Démonstration: Détection de biais

In [ ]:
# Installation
!pip install -q pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

# Simulation d'un modèle IA avec biais
np.random.seed(42)

# Création d'un dataset simulé
n_patients = 1000

data = pd.DataFrame({
    'patient_id': range(n_patients),
    'age': np.random.randint(18, 85, n_patients),
    'sexe': np.random.choice(['H', 'F'], n_patients),
    'ethnicite': np.random.choice(['Caucasien', 'Africain', 'Asiatique', 'Latino'], 
                                  n_patients, p=[0.7, 0.1, 0.1, 0.1]),  # Dataset déséquilibré!
    'pathologie_reelle': np.random.choice([0, 1], n_patients, p=[0.7, 0.3])
})

# Simulation de prédictions IA avec biais
def predire_avec_biais(row):
    """
    Simule un modèle IA qui performe mieux sur les Caucasiens
    car ils étaient sur-représentés dans les données d'entraînement.
    """
    verite = row['pathologie_reelle']
    
    # Performance selon l'ethnicité (simulée)
    if row['ethnicite'] == 'Caucasien':
        accuracy = 0.90  # 90% de précision
    else:
        accuracy = 0.70  # Seulement 70% pour les autres
    
    # Génération de la prédiction
    if np.random.random() < accuracy:
        return verite  # Prédiction correcte
    else:
        return 1 - verite  # Prédiction incorrecte

data['prediction_ia'] = data.apply(predire_avec_biais, axis=1)

print("Dataset simulé créé avec biais ethnique")
print(f"Total patients: {len(data)}")
print(f"\nRépartition ethnique:")
print(data['ethnicite'].value_counts())

In [ ]:
# Analyse de la performance par groupe
print("="*70)
print("ANALYSE DE PERFORMANCE PAR GROUPE ETHNIQUE")
print("="*70)

for ethnie in data['ethnicite'].unique():
    subset = data[data['ethnicite'] == ethnie]
    acc = accuracy_score(subset['pathologie_reelle'], subset['prediction_ia'])
    n = len(subset)
    
    print(f"\n{ethnie:12s} (n={n:3d}): Précision = {acc*100:.1f}%")

# Performance globale
acc_globale = accuracy_score(data['pathologie_reelle'], data['prediction_ia'])
print(f"\n{'GLOBALE':12s} (n={len(data):3d}): Précision = {acc_globale*100:.1f}%")
print("="*70)

print("\n OBSERVATION: Le modèle performe bien globalement (85%),")
print(" mais cette performance cache une INÉQUITÉ importante!")
print(" Les patients non-caucasiens reçoivent des soins de moins bonne qualité.")

In [ ]:
# Visualisation du biais
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1: Répartition ethnique du dataset
ethnie_counts = data['ethnicite'].value_counts()
axes[0].bar(ethnie_counts.index, ethnie_counts.values, color=['blue', 'orange', 'green', 'red'], alpha=0.7)
axes[0].set_title('Répartition Ethnique du Dataset', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Nombre de patients')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Graphique 2: Performance par groupe
performance_par_ethnie = []
for ethnie in data['ethnicite'].unique():
    subset = data[data['ethnicite'] == ethnie]
    acc = accuracy_score(subset['pathologie_reelle'], subset['prediction_ia']) * 100
    performance_par_ethnie.append({'Ethnicité': ethnie, 'Précision (%)': acc})

perf_df = pd.DataFrame(performance_par_ethnie)
colors_perf = ['green' if p > 85 else 'orange' if p > 75 else 'red' for p in perf_df['Précision (%)']]
axes[1].bar(perf_df['Ethnicité'], perf_df['Précision (%)'], color=colors_perf, alpha=0.7)
axes[1].axhline(y=85, color='blue', linestyle='--', label='Performance globale')
axes[1].set_title('Performance du Modèle par Groupe', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Précision (%)')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 LEÇON IMPORTANTE:")
print("   Une performance globale élevée peut cacher des inéquités graves.")
print("   Il faut TOUJOURS analyser la performance par sous-groupe!")

## 3. Cas Pratiques et Études de Cas

### Cas 1: L'algorithme qui aggrave les inégalités raciales

**Contexte réel (États-Unis, 2019):**
Un algorithme utilisé pour prioriser les soins de millions de patients défavorisait systématiquement les patients noirs.

**Le problème:**
- L'algorithme prédisait les "coûts futurs de santé"
- Hypothèse: coût élevé = besoin de soins élevé
- MAIS: Les patients noirs avaient historiquement moins accès aux soins
- Résultat: Coûts plus faibles → prédiction de "moindre besoin" → encore moins de soins!

**Leçon:** Un proxy (coût) n'est pas toujours une bonne mesure du vrai objectif (besoin de soins).

### Cas 2: L'IA qui détecte... l'hôpital au lieu de la maladie

**Contexte:**
Un modèle de détection de pneumonie sur radiographies thoraciques.

**Le problème découvert:**
- Performance excellente en validation (95%)
- Performance médiocre dans un autre hôpital (65%)
- Investigation: Le modèle avait appris à reconnaître les marqueurs métalliques spécifiques de certains hôpitaux!
- Ces marqueurs étaient corrélés (par hasard) avec la pneumonie dans le dataset

**Leçon:** L'IA peut apprendre des corrélations non pertinentes. La validation externe est cruciale.

### Cas 3: Le chatbot médical qui donne de mauvais conseils

**Contexte:**
Chatbot IA pour triage médical en ligne.

**Problèmes rapportés:**
- Minimise des symptômes sérieux chez certains patients
- Recommande des examens inutiles chez d'autres
- Pas de traçabilité du raisonnement
- En cas d'erreur: qui est responsable?

**Leçon:** L'IA médicale doit être traçable, explicable et supervisée.

## 4. Questions de Discussion

### Discussion 1: Consentement et transparence

**Scénario:** 
Un patient vient pour une radiographie. L'IA pré-analyse l'image et alerte le radiologue d'une possible tumeur.

**Questions:**
1. Le patient doit-il être informé qu'une IA est utilisée?
2. Doit-il pouvoir refuser l'analyse IA?
3. Si l'IA détecte quelque chose que le radiologue ne voit pas, que faire?
4. Si le radiologue voit quelque chose que l'IA a manqué, cela compte comme erreur de l'IA?

### Discussion 2: Responsabilité en cas d'erreur

**Scénario:**
L'IA recommande un traitement. Le médecin suit la recommandation. Le patient développe des complications graves.

**Questions:**
1. Qui est responsable: le médecin, l'entreprise IA, l'hôpital?
2. Le médecin peut-il se justifier en disant "l'IA a dit de faire ça"?
3. Quelle formation minimale le médecin doit-il avoir sur l'IA qu'il utilise?

### Discussion 3: Accès et équité

**Scénario:**
Une IA révolutionnaire pour le diagnostic précoce du cancer coûte 10,000€ par patient.

**Questions:**
1. Doit-elle être remboursée par la sécurité sociale?
2. Si oui, comment gérer les pays à faible revenu?
3. Est-il éthique d'utiliser une IA qui creuse les inégalités de santé?
4. Alternative: IA moins performante mais gratuite et accessible à tous?

## 5. Cadre pour une Utilisation Éthique de l'IA Médicale

### 5.1 Avant le déploiement

**Validation rigoureuse:**
- [ ] Études cliniques prospectives
- [ ] Validation sur populations diverses
- [ ] Analyse des biais potentiels
- [ ] Comparaison avec le standard actuel
- [ ] Évaluation de l'impact sur le workflow

**Transparence:**
- [ ] Données d'entraînement documentées
- [ ] Limitations clairement communiquées
- [ ] Explicabilité (pourquoi cette prédiction?)
- [ ] Processus de certification/régulation

### 5.2 Pendant l'utilisation

**Formation:**
- [ ] Formation des professionnels de santé
- [ ] Compréhension des limites de l'IA
- [ ] Savoir quand ne PAS faire confiance à l'IA

**Supervision:**
- [ ] Décision finale toujours humaine
- [ ] Monitoring de la performance en conditions réelles
- [ ] Système de signalement d'erreurs

**Respect du patient:**
- [ ] Information sur l'utilisation d'IA
- [ ] Droit de refuser
- [ ] Protection des données
- [ ] Maintien de la relation humaine médecin-patient

### 5.3 Après le déploiement

**Amélioration continue:**
- [ ] Collecte de retours d'expérience
- [ ] Mise à jour régulière du modèle
- [ ] Adaptation aux nouvelles données
- [ ] Réévaluation périodique

**Équité:**
- [ ] Surveillance des disparités de performance
- [ ] Correction des biais émergents
- [ ] Accès équitable à la technologie

## 6. Exercice Pratique: Analyse Éthique d'un Cas

**Cas proposé:**
Votre hôpital envisage d'adopter une IA de triage aux urgences qui prédit quels patients ont besoin d'être vus en priorité.

**Informations:**
- Performance: 88% de précision globale
- Entraîné sur 100,000 patients européens
- Coût: 50,000€/an
- Promet de réduire le temps d'attente de 30%

**Votre mission:**
Analysez ce cas selon les dimensions éthiques.

### Questions à se poser:

In [ ]:
# Framework d'analyse éthique
analyse_ethique = {
    'Bienfaisance': [
        "Bénéfice attendu: Réduction du temps d'attente",
        "Question: La réduction de 30% est-elle prouvée en conditions réelles?",
        "Question: Quel impact sur la qualité des soins?",
        "À investiguer: Études cliniques indépendantes?"
    ],
    'Non-malfaisance': [
        "Risque: Patients graves non priorisés (faux négatifs)",
        "Risque: Surcharge de patients non graves (faux positifs)",
        "Question: Quel est le taux d'erreur acceptable?",
        "À investiguer: Performance sur cas critiques rares?"
    ],
    'Autonomie': [
        "Question: Les patients sont-ils informés du triage IA?",
        "Question: Peuvent-ils demander un triage humain?",
        "Question: Comment sont utilisées leurs données?",
        "À investiguer: Politique de consentement?"
    ],
    'Justice': [
        "Problème: Dataset européen → biais géographique?",
        "Question: Performance égale selon âge, sexe, ethnicité?",
        "Question: Coût justifié vs embauche d'infirmiers de triage?",
        "À investiguer: Impact sur populations vulnérables?"
    ],
    'Transparence': [
        "Question: Comment l'IA décide-t-elle de la priorité?",
        "Question: Le personnel peut-il comprendre/contester?",
        "À investiguer: Algorithme explicable?"
    ],
    'Responsabilité': [
        "Question: Qui est responsable si erreur grave?",
        "Question: Formation du personnel prévue?",
        "À investiguer: Assurance et responsabilité légale?"
    ]
}

# Affichage de l'analyse
print("="*70)
print("ANALYSE ÉTHIQUE: IA DE TRIAGE AUX URGENCES")
print("="*70)

for dimension, questions in analyse_ethique.items():
    print(f"\n{dimension.upper()}:")
    for q in questions:
        print(f"  • {q}")

print("\n" + "="*70)
print("RECOMMANDATION PRÉLIMINAIRE:")
print("="*70)
print("""
Avant adoption, nous recommandons:

1. VALIDATION EXTERNE sur notre population locale
2. ANALYSE DES BIAIS par sous-groupes de patients
3. PÉRIODE DE TEST avec double triage (IA + humain)
4. FORMATION complète du personnel
5. PROTOCOLE CLAIR de responsabilité
6. INFORMATION des patients
7. COMITÉ D'ÉTHIQUE pour supervision continue

Coût estimé de validation: 20,000€
Durée: 6 mois

Seulement après validation positive: déploiement progressif avec monitoring.
""")
print("="*70)

## 7. Résumé et Principes Clés

### Les 10 Commandements de l'IA Médicale Éthique

1. **Tu valideras rigoureusement** avant tout déploiement clinique

2. **Tu analyseras les biais** dans tes données et ton modèle

3. **Tu respecteras l'autonomie** du patient (information, consentement)

4. **Tu maintiendras la supervision humaine** - L'IA aide, ne décide pas

5. **Tu assureras l'équité** - Performance égale pour tous les groupes

6. **Tu seras transparent** sur les limites et incertitudes

7. **Tu formeras les professionnels** à utiliser l'IA correctement

8. **Tu protégeras les données** des patients

9. **Tu monitoreras en continu** la performance réelle

10. **Tu assumeras la responsabilité** - Pas de "c'est la faute de l'IA"

### Quand NE PAS utiliser l'IA médicale?

- Quand elle n'a pas été validée sur votre population
- Quand vous ne comprenez pas son fonctionnement
- Quand elle remplace (au lieu d'assister) le jugement clinique
- Quand elle aggrave les inégalités
- Quand la responsabilité n'est pas claire
- Quand le patient refuse

### L'avenir de l'IA médicale éthique

**Vision positive:**
- IA comme assistant intelligent du médecin
- Détection précoce accessible à tous
- Réduction des inégalités de santé
- Plus de temps pour la relation médecin-patient

**Mais nécessite:**
- Régulation stricte et adaptée
- Formation généralisée
- Recherche éthique continue
- Dialogue sociétal sur les valeurs

## 8. Ressources pour Aller Plus Loin

### Lectures recommandées:

1. **WHO - Ethics and Governance of AI for Health** (2021)
   - Guide complet de l'OMS sur l'éthique de l'IA médicale

2. **Topol E. - Deep Medicine** (2019)
   - Vision humaniste de l'IA en médecine

3. **Obermeyer et al. - "Dissecting racial bias in an algorithm"** (Science, 2019)
   - Étude de cas importante sur les biais

### Réglementations à connaître:

- **RGPD** (Europe): Protection des données de santé
- **AI Act** (Europe): Régulation de l'IA, incluant applications médicales
- **FDA guidelines** (USA): Validation des dispositifs médicaux IA
- **Comités d'éthique** locaux: Consultation obligatoire

### Questions de réflexion finale:

1. Dans 20 ans, quel rôle l'IA devrait-elle avoir en médecine?
2. Comment maintenir l'humanité de la médecine à l'ère de l'IA?
3. Qui doit décider des limites éthiques de l'IA médicale?
4. Comment former les futurs médecins à travailler avec l'IA?

## 9. Projet Final - Proposition

### Votre Mission:

Proposez un projet d'IA médicale éthique et responsable.

**Format:**
- Présentation de 10 minutes
- Démonstration (optionnel)
- Document écrit (5 pages)

**Contenu attendu:**

1. **Problème clinique** identifié
2. **Solution IA** proposée
3. **Analyse éthique** complète
4. **Plan de validation**
5. **Stratégie de déploiement**
6. **Monitoring et amélioration**

**Critères d'évaluation:**
- Pertinence clinique (20%)
- Faisabilité technique (20%)
- Rigueur de l'analyse éthique (30%)
- Originalité (15%)
- Qualité de la présentation (15%)

Bon courage!

## 7. Points Clés

### Principes éthiques:
1. Validation rigoureuse avant déploiement
2. Analyser les biais (performance par sous-groupe)
3. Respecter l'autonomie du patient (information, consentement)
4. Maintenir supervision humaine - L'IA aide, ne décide pas
5. Assurer l'équité (pas de discrimination)
6. Transparence sur limites et incertitudes

### Quand NE PAS utiliser l'IA:
- Non validée sur votre population
- Vous ne comprenez pas son fonctionnement
- Remplace (au lieu d'assister) le jugement clinique
- Aggrave les inégalités
- Responsabilité non claire

### Message final:
L'IA médicale peut améliorer les soins, mais nécessite:
- Régulation stricte
- Formation des professionnels
- Dialogue sociétal sur les valeurs
- Vigilance continue sur les biais